# 00 — Data-generating process design

**Purpose.** Document the synthetic worlds used before bQuant data is available: the mathematical objects, the economic interpretation, and the validation logic.

This notebook is not the trading research itself. It is the model-design record that explains what the later notebooks are allowed to assume about synthetic data.

## Research question

We are interested in whether an illiquid long-tenor swap rate, currently scoped to **EUR 50Y IRS**, contains intraday level mean reversion that could justify delaying execution for a short horizon.

The synthetic layer therefore has two jobs:

1. Create a world where the thesis is true.
2. Create adversarial worlds where parts of the thesis are false.

A method is useful only if it works in the first world and fails gracefully in the adversarial worlds.

## Data contract

All DGPs return the same object:

$$
Y = \{Y_t : t = 0, 1, \ldots, T-1\},
$$

where $Y_t$ is the observed EUR 50Y swap-rate level at one-minute frequency.

Current contract:

- one column: `50Y`;
- timezone-aware CET `DatetimeIndex`;
- one-minute base grid;
- half-open session $[08{:}00, 17{:}00)$, so $T = 540$ observations;
- values are decimal rate levels, e.g. $0.028 = 2.8\%$.

We work in **level space**, not log space, because swap PnL and execution slippage are naturally measured in basis points.

## The OU process

OU means **Ornstein-Uhlenbeck**. It is the continuous-time Gaussian mean-reverting process

$$
dX_t = -\theta (X_t - \mu)\,dt + \sigma_{\mathrm{eff}}\,dW_t.
$$

Interpretation:

- $X_t$ is the efficient, unobserved swap-rate level.
- $\mu$ is the long-run intraday anchor level.
- $\theta > 0$ is the speed of pullback toward $\mu$.
- $\sigma_{\mathrm{eff}}$ is the efficient-price volatility.
- $W_t$ is Brownian motion.

If $X_t > \mu$, the drift term is negative. If $X_t < \mu$, the drift term is positive. That is the mean-reversion mechanism.

## Half-life

The conditional mean of the OU process satisfies

$$
\mathbb{E}[X_{t+h} - \mu \mid X_t]
= e^{-\theta h}(X_t - \mu).
$$

The half-life $h_{1/2}$ is the horizon where the expected deviation has halved:

$$
e^{-\theta h_{1/2}} = \frac{1}{2}.
$$

Solving gives

$$
h_{1/2} = \frac{\log 2}{\theta}.
$$

In the starter DGP we use $h_{1/2}=60$ minutes, so

$$
\theta = \frac{\log 2}{60}.
$$

This is a modelling assumption, not a calibrated fact. It is deliberately documented as low-confidence until real bQuant data is available.

## Exact one-minute discretisation

For a one-minute grid, the exact OU transition is

$$
X_{t+1}
= \mu + \phi (X_t - \mu) + \eta_{t+1},
$$

where

$$
\phi = e^{-\theta},
$$

and

$$
\eta_{t+1} \sim \mathcal{N}\left(
0,
\sigma_{\mathrm{eff}}^2 \frac{1 - e^{-2\theta}}{2\theta}
\right).
$$

This is what the code uses. We are not using an Euler approximation for the OU process.

## Microstructure noise

We observe a quote or executable mid, not the efficient rate itself. The observed rate is modelled as

$$
Y_t = X_t + \varepsilon_t,
$$

with

$$
\varepsilon_t \sim \mathcal{N}(0, \sigma_{\mathrm{noise}}^2),
\qquad
\varepsilon_t \perp \varepsilon_s \text{ for } t \ne s.
$$

This captures bid-ask bounce, stale quotes, quote shading, and other illiquidity effects in a minimal additive-noise form.

## Why iid quote noise creates negative return autocorrelation

The noise process $\varepsilon_t$ is iid in levels, but observed **returns** inherit negative autocorrelation because returns difference the noise:

$$
\Delta Y_t = Y_t - Y_{t-1}
= \Delta X_t + \varepsilon_t - \varepsilon_{t-1}.
$$

Ignore $\Delta X_t$ for the moment and define the noise return component

$$
u_t = \varepsilon_t - \varepsilon_{t-1}.
$$

Then

$$
\operatorname{Var}(\nu_t)
= \operatorname{Var}(\varepsilon_t - \varepsilon_{t-1})
= 2\sigma_{\mathrm{noise}}^2,
$$

and

$$
\operatorname{Cov}(\nu_t, \nu_{t-1})
= \operatorname{Cov}(\varepsilon_t - \varepsilon_{t-1},\varepsilon_{t-1} - \varepsilon_{t-2})
= -\sigma_{\mathrm{noise}}^2.
$$

So the pure-noise first-order autocorrelation is

$$
\rho_1
= \frac{-\sigma_{\mathrm{noise}}^2}{2\sigma_{\mathrm{noise}}^2}
= -\frac{1}{2}.
$$

With an efficient-price component included, the observed autocorrelation is less negative in magnitude, but the sign remains the standard microstructure signature when quote noise is material.

## DGP 1: thesis world — OU with noise

File: `synthetic/want/ou_with_noise.py`

Efficient process:

$$
X_{t+1} = \mu + e^{-\theta}(X_t - \mu) + \eta_{t+1}.
$$

Observed process:

$$
Y_t = X_t + \varepsilon_t.
$$

This world contains both things we want to study:

1. true level mean reversion in $X_t$;
2. microstructure contamination in $Y_t$.

A valid research procedure should detect microstructure noise at fine sampling intervals and should later detect level mean reversion once an appropriate candle granularity has been chosen.

## DGP 2: mean-reversion null — random walk with noise

File: `synthetic/dont_want/random_walk_with_noise.py`

Efficient process:

$$
X_{t+1} = X_t + u_{t+1},
\qquad
u_{t+1} \sim \mathcal{N}(0, \sigma_{\mathrm{eff}}^2).
$$

Observed process:

$$
Y_t = X_t + \varepsilon_t.
$$

This world has microstructure noise, but no true level mean reversion. It is the guardrail against confusing bid-ask bounce or noisy quotes with a tradable mean-reverting signal.

A mean-reversion method that rejects too often here is not usable.

## DGP 3: microstructure-noise null — OU without noise

File: `synthetic/dont_want/ou_no_noise.py`

Efficient process:

$$
X_{t+1} = \mu + e^{-\theta}(X_t - \mu) + \eta_{t+1}.
$$

Observed process:

$$
Y_t = X_t.
$$

This world has level mean reversion, but no microstructure-noise overlay. It is the guardrail for the RV/BV signature plot.

A microstructure-noise diagnostic should produce an approximately flat signature plot here. If it still finds a strong fine-grid RV inflation, the diagnostic is mistaking ordinary efficient-price dynamics for quote noise.

## Parameter choices

Starter values used in the synthetic notebook layer:

| Parameter | Value | Meaning |
|---|---:|---|
| $\mu$ | $0.028$ | representative EUR 50Y level, 2.8% |
| $\theta$ | $\log(2)/60$ | 60-minute half-life |
| $\sigma_{\mathrm{eff}}$ | $3.8\times 10^{-5}$ | efficient volatility, about 0.38 bp per $\sqrt{\mathrm{min}}$ |
| $\sigma_{\mathrm{noise}}$ | $1.5\times 10^{-5}$ | quote-noise standard deviation, about 0.15 bp |
| $T$ | $540$ | one trading session, 08:00 to 17:00 excluding 17:00 |

These are literature-grounded starter values, not final empirical estimates. The detailed source discussion lives in `docs/PARAMETER_SOURCES.md`.

## Validation principle

Validation is Monte Carlo, not single-path.

For a detector $D(\cdot)$ that returns 1 when it claims to find the target property:

Power on the thesis DGP:

$$
\widehat{\Pr}(D(Y)=1 \mid Y \sim \texttt{want}) \ge \beta.
$$

False-positive control on the null DGP:

$$
\widehat{\Pr}(D(Y)=1 \mid Y \sim \texttt{dont\_want}) \le \alpha.
$$

Current project defaults:

$$
N=500, \qquad \alpha=0.05, \qquad \beta=0.90.
$$

Automated tests allow a finite-sample tolerance of $1.5\alpha$ for false-positive rates.

## Why this is enough before bQuant

The current DGPs deliberately separate three effects:

| Effect | Present in OU + noise | Present in random walk + noise | Present in OU no noise |
|---|---:|---:|---:|
| Level mean reversion | yes | no | yes |
| Microstructure noise | yes | yes | no |
| Negative short-lag return autocorrelation from quote noise | yes | yes | no |

This lets the notebook series ask clean questions:

1. **Layer 1:** At what sampling granularity does microstructure noise stop dominating RV/BV diagnostics?
2. **Layer 2:** At that chosen granularity, is there evidence of level mean reversion?
3. **Layer 3:** If yes, is the effect large enough to justify delaying execution after costs and liquidity constraints?

So yes: after this design record, the next real notebook should be the RV/BV signature-plot notebook.

In [ ]:
from math import log

import pandas as pd

from synthetic.want.ou_with_noise import simulate as simulate_ou_noise
from synthetic.dont_want.random_walk_with_noise import simulate as simulate_rw_noise
from synthetic.dont_want.ou_no_noise import simulate as simulate_ou_no_noise

START = pd.Timestamp('2024-01-02 08:00', tz='CET')
PARAMS = {
    'mu': 0.028,
    'theta': log(2) / 60,
    'sigma_eff': 3.8e-5,
    'sigma_noise': 1.5e-5,
    'n_minutes': 540,
    'start': START,
    'seed': 0,
}

ou_noise = simulate_ou_noise(**PARAMS)
rw_noise = simulate_rw_noise(
    start_level=PARAMS['mu'],
    sigma_eff=PARAMS['sigma_eff'],
    sigma_noise=PARAMS['sigma_noise'],
    n_minutes=PARAMS['n_minutes'],
    start=START,
    seed=0,
)
ou_no_noise = simulate_ou_no_noise(
    mu=PARAMS['mu'],
    theta=PARAMS['theta'],
    sigma_eff=PARAMS['sigma_eff'],
    n_minutes=PARAMS['n_minutes'],
    start=START,
    seed=0,
)

summary = pd.DataFrame(
    {
        'OU + noise': ou_noise['50Y'].diff().autocorr(lag=1),
        'random walk + noise': rw_noise['50Y'].diff().autocorr(lag=1),
        'OU no noise': ou_no_noise['50Y'].diff().autocorr(lag=1),
    },
    index=['lag-1 autocorr of 1-minute rate changes'],
).T

summary